# Model 2: All Pre-Transfer Qualities — Midfielders

**Can ALL pre-transfer qualities together predict each post-transfer quality?**

$$\text{to}_{Q_i} = \alpha + \sum_{j=1}^{17} \beta_j \cdot \text{from}_{Q_j}$$

Now each quality is predicted using **all 17 pre-transfer qualities** as features (cross-quality effects).

- **Input:** 17 features (all pre-transfer qualities)
- **Output:** 1 post-transfer quality per model (17 models total)
- **Position:** Midfielders only
- **Split:** Train 2018–2023, Test 2024

Compared to Model 1 (each quality predicts only itself), this captures **interactions** — e.g., does a midfielder's pre-transfer Pressing help predict their post-transfer Involvement?

---

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

DATA = "../../../../thesis_data/processed_data/thesis_model_dataset/active/within_league_transfers_v5.parquet"
df = pd.read_parquet(DATA)
mf = df[df["from_position"] == "Midfielder"].copy()
print(f"Midfielders: {len(mf):,} rows")

Midfielders: 5,230 rows


In [2]:
QUALITIES = [
    "Active defence", "Aerial threat", "Box threat", "Composure",
    "Defensive heading", "Dribbling", "Effectiveness", "Finishing",
    "Hold-up play", "Intelligent defence", "Involvement",
    "Passing quality", "Pressing", "Progression",
    "Providing teammates", "Run quality", "Winning duels",
]

from_cols = [f"from_{q}" for q in QUALITIES]
to_cols   = [f"to_{q}" for q in QUALITIES]

mf_clean = mf[from_cols + to_cols + ["from_season"]].dropna()
train = mf_clean[mf_clean["from_season"] <= 2023]
test  = mf_clean[mf_clean["from_season"] == 2024]
print(f"Clean: {len(mf_clean):,} | Train: {len(train):,} | Test: {len(test):,}")

Clean: 5,159 | Train: 4,692 | Test: 467


## Cross-Quality Correlations (Pre-Transfer)

How correlated are the 17 pre-transfer qualities? High correlations between features may cause multicollinearity.

In [3]:
corr = train[from_cols].corr()
labels = QUALITIES  # clean names for display

fig = px.imshow(corr.values, x=labels, y=labels, color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1, text_auto=".2f", aspect="auto")
fig.update_layout(title="Pre-Transfer Quality Correlations (Midfielders, train set)",
    height=650, template="plotly_white")
fig.show()

---
## Results

For each quality: `to_Qi = α + Σ βj · from_Qj` (17 features per model).

In [4]:
results = []
models = {}

for q in QUALITIES:
    X_train = sm.add_constant(train[from_cols])
    X_test  = sm.add_constant(test[from_cols])
    y_train = train[f"to_{q}"]
    y_test  = test[f"to_{q}"]

    model = sm.OLS(y_train, X_train).fit()
    y_pred = model.predict(X_test)

    results.append({
        "Quality": q,
        "R²_train": model.rsquared,
        "R²_adj": model.rsquared_adj,
        "R²_test": r2_score(y_test, y_pred),
        "MAE_test": mean_absolute_error(y_test, y_pred),
        "F_pvalue": model.f_pvalue,
    })
    models[q] = model

res = pd.DataFrame(results)
print(res.to_string(index=False))
print(f"\nMean R²_train: {res['R²_train'].mean():.4f}")
print(f"Mean R²_adj:   {res['R²_adj'].mean():.4f}")
print(f"Mean R²_test:  {res['R²_test'].mean():.4f}")
print(f"Mean MAE_test: {res['MAE_test'].mean():.4f}")

            Quality  R²_train  R²_adj  R²_test  MAE_test  F_pvalue
     Active defence    0.3820  0.3797   0.3453    0.4314    0.0000
      Aerial threat    0.3685  0.3663   0.2522    0.4310    0.0000
         Box threat    0.4339  0.4318   0.3609    0.3638    0.0000
          Composure    0.1500  0.1469   0.0755    0.3856    0.0000
  Defensive heading    0.4187  0.4166   0.4032    0.4749    0.0000
          Dribbling    0.3499  0.3475   0.2041    0.3319    0.0000
      Effectiveness    0.2739  0.2712   0.2317    0.2447    0.0000
          Finishing    0.0785  0.0752   0.0411    0.5261    0.0000
       Hold-up play    0.2610  0.2583   0.1866    0.2566    0.0000
Intelligent defence    0.3756  0.3733   0.3432    0.4485    0.0000
        Involvement    0.3243  0.3218   0.3603    0.3682    0.0000
    Passing quality    0.4875  0.4857   0.4663    0.4017    0.0000
           Pressing    0.3362  0.3338   0.3323    0.4478    0.0000
        Progression    0.4469  0.4449   0.4134    0.4744    0.

### Comparison vs Model 1 (Naive Baseline)

Does adding cross-quality features actually help?

In [5]:
# Model 1 results (from notebook 01)
m1_results = []
for q in QUALITIES:
    X_tr = sm.add_constant(train[[f"from_{q}"]])
    X_te = sm.add_constant(test[[f"from_{q}"]])
    m = sm.OLS(train[f"to_{q}"], X_tr).fit()
    m1_results.append({
        "Quality": q,
        "R²_test_M1": r2_score(test[f"to_{q}"], m.predict(X_te)),
    })

m1 = pd.DataFrame(m1_results)
comp = res[["Quality", "R²_test"]].merge(m1, on="Quality")
comp["Δ R²_test"] = comp["R²_test"] - comp["R²_test_M1"]
comp = comp.rename(columns={"R²_test": "R²_test_M2"})
print(comp.to_string(index=False))
print(f"\nMean Δ R²_test: {comp['Δ R²_test'].mean():.4f}")
print(f"Qualities improved: {(comp['Δ R²_test'] > 0).sum()} / {len(comp)}")

            Quality  R²_test_M2  R²_test_M1  Δ R²_test
     Active defence      0.3453      0.2999     0.0454
      Aerial threat      0.2522      0.1867     0.0655
         Box threat      0.3609      0.3421     0.0188
          Composure      0.0755      0.0256     0.0498
  Defensive heading      0.4032      0.3576     0.0456
          Dribbling      0.2041      0.1895     0.0146
      Effectiveness      0.2317      0.1445     0.0872
          Finishing      0.0411      0.0135     0.0276
       Hold-up play      0.1866      0.1301     0.0565
Intelligent defence      0.3432      0.3037     0.0394
        Involvement      0.3603      0.3510     0.0093
    Passing quality      0.4663      0.4484     0.0178
           Pressing      0.3323      0.2883     0.0440
        Progression      0.4134      0.3815     0.0319
Providing teammates      0.3990      0.3596     0.0395
        Run quality      0.3997      0.3672     0.0326
      Winning duels      0.2188      0.1214     0.0975

Mean Δ R²

In [6]:
fig = go.Figure()
fig.add_trace(go.Bar(name="Model 1 (self only)", x=comp["Quality"], y=comp["R²_test_M1"],
    marker_color="#636EFA"))
fig.add_trace(go.Bar(name="Model 2 (all 17 pre-Q)", x=comp["Quality"], y=comp["R²_test_M2"],
    marker_color="#EF553B"))
fig.update_layout(title="R²_test — Model 1 vs Model 2 (Midfielders)",
    yaxis_title="R²_test", barmode="group", height=450, template="plotly_white",
    xaxis_tickangle=-45)
fig.show()

### Full OLS Summary — Best and Worst

In [7]:
best_q = res.loc[res["R²_test"].idxmax(), "Quality"]
worst_q = res.loc[res["R²_test"].idxmin(), "Quality"]

print(f"BEST: {best_q} (R²_test = {res.loc[res['R²_test'].idxmax(), 'R²_test']:.4f})")
print(models[best_q].summary())
print(f"\n{'='*70}")
print(f"WORST: {worst_q} (R²_test = {res.loc[res['R²_test'].idxmin(), 'R²_test']:.4f})")
print(models[worst_q].summary())

BEST: Passing quality (R²_test = 0.4663)
                            OLS Regression Results                            
Dep. Variable:     to_Passing quality   R-squared:                       0.488
Model:                            OLS   Adj. R-squared:                  0.486
Method:                 Least Squares   F-statistic:                     261.5
Date:                Wed, 11 Mar 2026   Prob (F-statistic):               0.00
Time:                        16:57:32   Log-Likelihood:                -3500.0
No. Observations:                4692   AIC:                             7036.
Df Residuals:                    4674   BIC:                             7152.
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------

### Coefficient Heatmap

Which pre-transfer qualities matter most for predicting each post-transfer quality? Shows β coefficients across all 17 models.

In [8]:
# Build coefficient matrix (exclude intercept)
coef_matrix = pd.DataFrame(index=QUALITIES, columns=QUALITIES, dtype=float)
for q in QUALITIES:
    params = models[q].params
    for fq in QUALITIES:
        coef_matrix.loc[q, fq] = params[f"from_{fq}"]

fig = px.imshow(coef_matrix.values.astype(float), x=QUALITIES, y=QUALITIES,
    color_continuous_scale="RdBu_r", zmin=-0.3, zmax=0.3, text_auto=".2f", aspect="auto")
fig.update_layout(title="OLS Coefficients: from_Qj → to_Qi (Midfielders)",
    xaxis_title="Feature (from_Q)", yaxis_title="Target (to_Q)",
    height=650, template="plotly_white")
fig.show()

---
## Takeaway

Usar las 17 qualities pre-transfer como features (en vez de solo la misma quality) mejora ligeramente el R²_test.

**La diagonal domina** — cada quality sigue siendo su mejor predictor, pero hay cross-effects interesantes (e.g., from_Pressing ayuda a predecir to_Active defence).

**Siguiente modelo:** Agregar team qualities (estilo táctico del equipo origen y destino) como features adicionales.